<table style="width:100%">
<tr>
<td style="vertical-align:middle; text-align:left;">
<font size="2">
<a href="http://mng.bz/orYv">처음부터 만드는 대형 언어 모델</a> 책의 보조 코드 by <a href="https://sebastianraschka.com">Sebastian Raschka</a><br>
<br>코드 저장소: <a href="https://github.com/rasbt/LLMs-from-scratch">https://github.com/rasbt/LLMs-from-scratch</a>
</font>
</td>
<td style="vertical-align:middle; text-align:left;">
<a href="http://mng.bz/orYv"><img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/cover-small.webp" width="100px"></a>
</td>
</tr>
</table>

# 2장 연습 문제 해답

이 노트북에서 사용되는 패키지들:

In [ ]:
from importlib.metadata import version

print("torch version:", version("torch"))
print("tiktoken version:", version("tiktoken"))

# 연습 문제 2.1

In [ ]:
import tiktoken

tokenizer = tiktoken.get_encoding("gpt2")

In [ ]:
integers = tokenizer.encode("Akwirw ier")
print(integers)

In [ ]:
for i in integers:
    print(f"{i} -> {tokenizer.decode([i])}")

In [ ]:
tokenizer.encode("Ak")

In [ ]:
tokenizer.encode("w")

In [ ]:
tokenizer.encode("ir")

In [ ]:
tokenizer.encode("w")

In [ ]:
tokenizer.encode(" ")

In [ ]:
tokenizer.encode("ier")

In [ ]:
tokenizer.decode([33901, 86, 343, 86, 220, 959])

# 연습 문제 2.2

In [ ]:
import tiktoken
import torch
from torch.utils.data import Dataset, DataLoader


class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []

        # 전체 텍스트를 토큰화합니다
        token_ids = tokenizer.encode(txt, allowed_special={"<|endoftext|>"})

        # 슬라이딩 윈도우를 사용하여 책을 max_length의 겹치는 시퀀스로 청킹합니다
        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i:i + max_length]
            target_chunk = token_ids[i + 1: i + max_length + 1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]


def create_dataloader(txt, batch_size=4, max_length=256, stride=128):
    # 토크나이저를 초기화합니다
    tokenizer = tiktoken.get_encoding("gpt2")

    # 데이터셋을 생성합니다
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)

    # 데이터로더를 생성합니다
    dataloader = DataLoader(dataset, batch_size=batch_size)

    return dataloader


with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

tokenizer = tiktoken.get_encoding("gpt2")
encoded_text = tokenizer.encode(raw_text)

In [ ]:
dataloader = create_dataloader(raw_text, batch_size=4, max_length=2, stride=2)

for batch in dataloader:
    x, y = batch
    break

x

In [ ]:
dataloader = create_dataloader(raw_text, batch_size=4, max_length=8, stride=2)

for batch in dataloader:
    x, y = batch
    break

x